# Merge financial and sporting data

This notebook merges the validated financial CORE with the processed sporting dataset and audits unmatched club-years before modeling.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

financial = pd.read_csv('../data/processed/core_financials.csv')
sporting = pd.read_csv('../data/processed/sporting_data.csv')

## Check merge keys

In [ ]:
financial_duplicates = financial[financial.duplicated(subset=['club', 'fiscal_year'], keep=False)]

sporting_duplicates = sporting[sporting.duplicated(subset=['club', 'season'], keep=False)]

assert financial_duplicates.empty
assert sporting_duplicates.empty

## Financial CORE left join sporting data

Financial club-years remain in the dataset even when the club had no national division in that season.

In [ ]:
merged = financial.merge(
    sporting,
    how='left',
    left_on=['club', 'fiscal_year'],
    right_on=['club', 'season'],
    validate='one_to_one',
    indicator=True,
)

merged['_merge'].value_counts()

_merge
both          333
left_only       6
right_only      0
Name: count, dtype: int64

## Audit financial club-years without sporting match

In [4]:
financial_without_sporting = (
    merged.loc[
        merged['_merge'] == 'left_only',
        ['club', 'fiscal_year'],
    ]
    .sort_values(['club', 'fiscal_year'])
    .reset_index(drop=True)
)

financial_without_sporting

,club,fiscal_year
0,Mirassol,2016
1,Mirassol,2017
2,Paraná Clube,2023
3,Paraná Clube,2024
4,Paraná Clube,2025
5,Santa Cruz,2024


## Full club-year coverage audit

This distinguishes financial-only, sporting-only and absent-from-both club-years across the financial club universe.

In [5]:
financial_clubs = sorted(financial['club'].unique())

first_year = min(financial['fiscal_year'].min(), sporting['season'].min())
last_year = max(financial['fiscal_year'].max(), sporting['season'].max())

panel = pd.MultiIndex.from_product(
    [financial_clubs, range(first_year, last_year + 1)],
    names=['club', 'year'],
).to_frame(index=False)

financial_keys = (
    financial[['club', 'fiscal_year']]
    .rename(columns={'fiscal_year': 'year'})
    .assign(financial_present=1)
)

sporting_keys = (
    sporting.loc[
        sporting['club'].isin(financial_clubs),
        ['club', 'season'],
    ]
    .rename(columns={'season': 'year'})
    .assign(sporting_present=1)
)

coverage = (
    panel
    .merge(financial_keys, how='left', on=['club', 'year'])
    .merge(sporting_keys, how='left', on=['club', 'year'])
)

coverage[['financial_present', 'sporting_present']] = (
    coverage[['financial_present', 'sporting_present']]
    .fillna(0)
    .astype(int)
)

coverage['status'] = np.select(
    [
        (coverage['financial_present'] == 1) & (coverage['sporting_present'] == 1),
        (coverage['financial_present'] == 1) & (coverage['sporting_present'] == 0),
        (coverage['financial_present'] == 0) & (coverage['sporting_present'] == 1),
    ],
    ['both', 'financial_only', 'sporting_only'],
    default='neither',
)

coverage['status'].value_counts()

status
both              333
sporting_only      10
financial_only      6
neither             1
Name: count, dtype: int64

In [6]:
coverage.loc[
    coverage['status'] != 'both'
].sort_values(['status', 'club', 'year']).reset_index(drop=True)

,club,year,financial_present,sporting_present,status
0,Mirassol,2016,1,0,financial_only
1,Mirassol,2017,1,0,financial_only
2,Paraná Clube,2023,1,0,financial_only
3,Paraná Clube,2024,1,0,financial_only
4,Paraná Clube,2025,1,0,financial_only
5,Santa Cruz,2024,1,0,financial_only
6,Mirassol,2019,0,0,neither
7,CSA,2025,0,1,sporting_only
8,Cuiabá,2016,0,1,sporting_only
9,Figueirense,2024,0,1,sporting_only


## Export merged dataset

`season` is redundant after the merge because it is equal to `fiscal_year` for matched rows. Missing sporting fields are intentionally preserved for financial club-years without a national division.

In [7]:
final = merged.drop(columns=['season', '_merge'])

assert len(final) == len(financial)
assert not final.duplicated(subset=['club', 'fiscal_year']).any()

output_path = Path('../data/processed/financial_sporting_data.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

final.to_csv(output_path, index=False)

In [8]:
pd.read_csv(output_path)

,club,fiscal_year,cash,current_assets,total_assets,current_liabilities,total_liabilities,equity,net_result,division,...,league_size,promoted,relegated,table_source,table_source_url,outcome_source,notes,points_per_game,goal_difference_per_game,relative_league_position
0,América-MG,2016,21.577,9676.933,221677.162,11340.124,75540.766,146136.395,9531.195,1.0,...,20.0,0.0,1.0,RSSSF Brasil,https://www.rsssfbrasil.com/tablesae/br2016.htm,RSSSF Brasil,Official CBF table unavailable for automated e...,0.736842,-0.921053,0.000000
1,América-MG,2017,125.368,13621.152,221205.956,25213.432,80229.944,140976.012,-5100.137,2.0,...,20.0,1.0,0.0,RSSSF Brasil,https://www.rsssfbrasil.com/tablesae/br2017l2.htm,RSSSF Brasil,Official CBF table unavailable for automated e...,1.921053,0.552632,1.000000
2,América-MG,2018,16.000,4331.000,220499.000,35054.000,88012.000,132487.000,-3836.000,1.0,...,20.0,0.0,1.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.052632,-0.447368,0.157895
3,América-MG,2019,3833.000,8343.000,216082.000,39144.000,106200.000,109882.000,-17545.238,2.0,...,20.0,0.0,0.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.605263,0.210526,0.789474
4,América-MG,2020,983.872,11662.137,217969.063,43494.365,116638.014,101331.049,-8550.982,2.0,...,20.0,1.0,0.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.921053,0.526316,0.947368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,Vitória,2021,120.000,8271.000,27966.000,160566.000,272288.000,-244322.000,-42320.000,2.0,...,20.0,0.0,1.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.052632,-0.026316,0.105263
335,Vitória,2022,14.000,11330.000,33900.000,108165.000,243397.000,-209497.000,34825.000,3.0,...,20.0,1.0,0.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.526316,0.315789,0.684211
336,Vitória,2023,55.000,7281.000,30857.000,118052.000,264443.000,-233586.000,-24089.000,2.0,...,20.0,1.0,0.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.894737,0.500000,1.000000
337,Vitória,2024,272.000,28625.000,70771.000,143386.000,345662.000,-274891.000,-41305.000,1.0,...,20.0,0.0,0.0,CBF,https://www.cbf.com.br/futebol-brasileiro/tabe...,CBF,NaN,1.236842,-0.184211,0.473684
